In [1]:
import pandas as pd
import numpy as np
data_path = 'D:/Datasets/UserBehavior.csv' 
# 读取前 200 万行数据
columns = ['user_id', 'item_id', 'category_id', 'behavior_type', 'timestamp']
df = pd.read_csv(data_path, names=columns, nrows=2_000_000)
# 打印数据形状和预览
print("数据形状:", df.shape)
df.head()

数据形状: (2000000, 5)


,user_id,item_id,category_id,behavior_type,timestamp
0,1,2268318,2520377,pv,1511544070
1,1,2333346,2520771,pv,1511561733
2,1,2576651,149192,pv,1511572885
3,1,3830808,4181361,pv,1511593493
4,1,4365585,2520377,pv,1511596146


In [2]:
# 1. 时间戳转换（转化为北京时间）
df['datetime'] = pd.to_datetime(df['timestamp'], unit='s') + pd.Timedelta(hours=8)

# 2. 提取日期和小时
df['date'] = df['datetime'].dt.date
df['hour'] = df['datetime'].dt.hour

# 3. 检查时间范围（必须落在 2017-11-25 到 2017-12-03 之间）
print("数据起始日期:", df['date'].min())
print("数据结束日期:", df['date'].max())

# 4. 检查重复行并去重
duplicate_count = df.duplicated().sum()
print(f"发现重复行数: {duplicate_count}")
df = df.drop_duplicates()

# 5. 行为类型分布统计（打个底，看下各类行为占比）
behavior_counts = df['behavior_type'].value_counts()
print("各行为分布情况：\n", behavior_counts)

# 6. 把行为映射为数字（为后面漏斗排序做准备）
behavior_map = {'pv': 1, 'fav': 2, 'cart': 3, 'buy': 4}
df['behavior_code'] = df['behavior_type'].map(behavior_map)

# 查看处理后的数据
df.head()

数据起始日期: 1970-01-01
数据结束日期: 2018-08-28
发现重复行数: 0
各行为分布情况：
 behavior_type
pv      1791216
cart     111015
fav       57526
buy       40243
Name: count, dtype: int64


,user_id,item_id,category_id,behavior_type,timestamp,datetime,date,hour,behavior_code
0,1,2268318,2520377,pv,1511544070,2017-11-25 01:21:10,2017-11-25,1,1
1,1,2333346,2520771,pv,1511561733,2017-11-25 06:15:33,2017-11-25,6,1
2,1,2576651,149192,pv,1511572885,2017-11-25 09:21:25,2017-11-25,9,1
3,1,3830808,4181361,pv,1511593493,2017-11-25 15:04:53,2017-11-25,15,1
4,1,4365585,2520377,pv,1511596146,2017-11-25 15:49:06,2017-11-25,15,1


In [9]:
# 1. 直接在原数据上新增三列布尔值（True/False）
df['is_pv'] = df['behavior_type'] == 'pv'
df['is_cart'] = df['behavior_type'] == 'cart'
df['is_buy'] = df['behavior_type'] == 'buy'

# 2. 按用户分组，求最大值。
# (在 Python 里，True 等于 1，False 等于 0。所以 max() 只要用户有过一次该行为，结果就是 True)
user_actions = df.groupby('user_id')[['is_pv', 'is_cart', 'is_buy']].max().reset_index()

# 3. 计算绝对漏斗（严格按链路）
step_1_pv = user_actions[user_actions['is_pv'] == True]
step_2_cart = step_1_pv[step_1_pv['is_cart'] == True]
step_3_buy = step_2_cart[step_2_cart['is_buy'] == True]

# 4. 打印结果
print("=== 严格转化漏斗 ===")
print(f"1. 浏览用户数 (pv): {len(step_1_pv)}")
print(f"2. 加购用户数 (浏览且加购): {len(step_2_cart)}")
print(f"3. 购买用户数 (浏览且加购且购买): {len(step_3_buy)}")

print("\n=== 转化率计算 ===")
print(f"浏览 -> 加购转化率: {len(step_2_cart) / len(step_1_pv) * 100:.2f}%")
print(f"加购 -> 购买转化率: {len(step_3_buy) / len(step_2_cart) * 100:.2f}%")
print(f"整体转化率: {len(step_3_buy) / len(step_1_pv) * 100:.2f}%")

=== 严格转化漏斗 ===
1. 浏览用户数 (pv): 19463
2. 加购用户数 (浏览且加购): 14598
3. 购买用户数 (浏览且加购且购买): 10500

=== 转化率计算 ===
浏览 -> 加购转化率: 75.00%
加购 -> 购买转化率: 71.93%
整体转化率: 53.95%


In [10]:
# 1. 先按 user_id 聚合基础的行为次数
user_df = df.groupby('user_id').agg(
    pv_count=('behavior_type', lambda x: (x == 'pv').sum()),
    fav_count=('behavior_type', lambda x: (x == 'fav').sum()),
    cart_count=('behavior_type', lambda x: (x == 'cart').sum()),
    buy_count=('behavior_type', lambda x: (x == 'buy').sum())
).reset_index()

# 2. 单独拎出购买数据，计算每个用户的最后一次购买日期（这样不会误伤没买过的人）
buy_df = df[df['behavior_type'] == 'buy']
last_buy = buy_df.groupby('user_id')['date'].max().reset_index()
last_buy.columns = ['user_id', 'last_buy_date']

# 3. 把最后一次购买日期，用 left join 贴回主表
user_df = pd.merge(user_df, last_buy, on='user_id', how='left')

# 4. 计算 R、F、M
analysis_date = pd.to_datetime('2017-12-03').date()
user_df['R'] = user_df['last_buy_date'].apply(lambda x: (analysis_date - x).days if pd.notna(x) else 99)
user_df['F'] = user_df['buy_count']
user_df['M'] = user_df['buy_count'] # 以购买次数作为 M 的代理

# 5. 过滤僵尸号
user_df = user_df[user_df['pv_count'] > 0]

print("修复后的用户宽表维度:", user_df.shape)
# 专门看看买了东西的人，R 值是否正常了
print("\n验证购买过的人群数据：")
display(user_df[user_df['buy_count'] > 0].head())

修复后的用户宽表维度: (19463, 9)

验证购买过的人群数据：


,user_id,pv_count,fav_count,cart_count,buy_count,last_buy_date,R,F,M
2,100,84,6,0,8,2017-11-28,5,8,8
4,117,156,0,4,10,2017-11-28,5,10,10
6,119,141,6,1,3,2017-11-29,4,3,3
7,121,99,7,13,1,2017-11-25,8,1,1
8,122,243,26,10,3,2017-12-02,1,3,3


In [13]:
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# 1. 筛选有购买记录的用户进行聚类
rfm_df = user_df[user_df['buy_count'] > 0].copy()

# 提取特征矩阵
X = rfm_df[['R', 'F', 'M']]

# 2. 数据标准化 (非常重要！R单位是天，F和M是次数，量级不同，必须消除量纲影响)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. 遍历 K=2 到 K=6，寻找最佳聚类数
print("=== 寻找最佳 K 值 ===")
best_k = 3 # 设定一个默认值
best_score = -1

for k in range(2, 7):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    
    # 计算轮廓系数（越接近1越好）
    score = silhouette_score(X_scaled, labels)
    print(f"K={k}, 轮廓系数(Silhouette Score) = {score:.4f}")
    
    if score > best_score:
        best_score = score
        best_k = k

print(f"\n✅ 根据轮廓系数，最佳聚类数为 K={best_k}")

# 4. 用最佳的 K 值进行最终聚类
kmeans_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
rfm_df['Cluster'] = kmeans_final.fit_predict(X_scaled)

# 5. 查看每个簇的 RFM 均值，为后面给用户打标签做准备
cluster_summary = rfm_df.groupby('Cluster')[['R', 'F', 'M']].mean().round(2)
cluster_summary['User_Count'] = rfm_df['Cluster'].value_counts()
print("\n=== 各聚类中心的 RFM 均值 ===")
print(cluster_summary)

=== 寻找最佳 K 值 ===
K=2, 轮廓系数(Silhouette Score) = 0.5484
K=3, 轮廓系数(Silhouette Score) = 0.4593
K=4, 轮廓系数(Silhouette Score) = 0.4875
K=5, 轮廓系数(Silhouette Score) = 0.4712
K=6, 轮廓系数(Silhouette Score) = 0.4431

✅ 根据轮廓系数，最佳聚类数为 K=2

=== 各聚类中心的 RFM 均值 ===
            R     F     M  User_Count
Cluster                              
0        2.73  2.22  2.22       11739
1        1.06  9.12  9.12        1524
